# Audio → LLM POC — Colab Trainer

This notebook pulls `Kesehet/audio-llm-poc`, prepares a small public speech corpus, and starts Stage-1 projector training.

**Recommended runtime:** GPU. A T4 is enough for the starter run; L4/A100 is better.

Stage 1 objective:

`audio → frozen Whisper encoder → trainable projector → frozen Qwen → transcription`


In [ ]:
!nvidia-smi
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
!rm -rf /content/audio-llm-poc
!git clone https://github.com/Kesehet/audio-llm-poc.git /content/audio-llm-poc
%cd /content/audio-llm-poc
!git log -1 --oneline


In [ ]:
!pip -q install -r requirements.txt


## Choose starter dataset size

The default is intentionally small so the first run proves the pipeline before spending hours on training.

Set `FAST_TEST = False` after the first successful run to use a larger starter corpus.


In [ ]:
FAST_TEST = True

if FAST_TEST:
    limits = {
        "fleurs-en": 150,
        "fleurs-hi": 150,
        "librispeech-clean": 300,
        "voxpopuli-en": 300,
    }
else:
    limits = {
        "fleurs-en": 1000,
        "fleurs-hi": 1000,
        "librispeech-clean": 3000,
        "voxpopuli-en": 3000,
    }

limits


In [ ]:
import subprocess, sys

for dataset_name, limit in limits.items():
    print(f"\n=== {dataset_name}: {limit} samples ===")
    subprocess.run(
        [sys.executable, "scripts/prepare_public_asr.py", dataset_name, "--limit", str(limit)],
        check=True,
    )


In [ ]:
!python scripts/merge_manifests.py   data/public/fleurs-en/manifest.jsonl   data/public/fleurs-hi/manifest.jsonl   data/public/librispeech-clean/manifest.jsonl   data/public/voxpopuli-en/manifest.jsonl   --output data/stage1.jsonl

!wc -l data/stage1.jsonl
!head -n 2 data/stage1.jsonl


## Colab training config

This uses FP16 for T4-class GPUs, batch size 1, and trains only the projector.


In [ ]:
from pathlib import Path
import yaml, torch

cfg = yaml.safe_load(Path("configs/poc.yaml").read_text())
cfg["mixed_precision"] = "fp16"
cfg["batch_size"] = 1
cfg["grad_accum_steps"] = 8
cfg["epochs"] = 3 if FAST_TEST else 2
cfg["output_dir"] = "checkpoints/colab-stage1"

# Smaller Whisper encoder if the assigned GPU is unusually constrained.
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram_gb < 13:
        cfg["audio_encoder"] = "openai/whisper-base"

Path("configs/colab-runtime.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(Path("configs/colab-runtime.yaml").read_text())


In [ ]:
!python train.py --manifest data/stage1.jsonl --config configs/colab-runtime.yaml


## Check saved projector weights


In [ ]:
from pathlib import Path
checkpoints = sorted(Path("checkpoints/colab-stage1").glob("projector-epoch-*.pt"))
print("Saved checkpoints:")
for p in checkpoints:
    print(" -", p, f"({p.stat().st_size / 1024**2:.1f} MB)")


## Optional: save checkpoints to Google Drive

Run this after training if you want the weights to survive the Colab runtime.


In [ ]:
# Uncomment to save to Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/audio-llm-poc-checkpoints
# !cp -v checkpoints/colab-stage1/*.pt /content/drive/MyDrive/audio-llm-poc-checkpoints/
